In [1]:
!python -V

Python 3.12.14


In [2]:
import pandas as pd

In [3]:
import pickle

In [4]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [5]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

2026/11/08 16:57:37 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1794153457853, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794153457853, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [6]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [7]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [8]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [9]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [10]:
import xgboost as xgb

In [11]:
from pathlib import Path

In [12]:
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [13]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=30,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/home/manwe_acer/mlops-zoomcamp/03-orchestration/code/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [17:03:29] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.44174
[1]	validation-rmse:10.76627
[2]	validation-rmse:10.17716
[3]	validation-rmse:9.66439
[4]	validation-rmse:9.21866
[5]	validation-rmse:8.83455
[6]	validation-rmse:8.50406
[7]	validation-rmse:8.22009
[8]	validation-rmse:7.97540
[9]	validation-rmse:7.76692
[10]	validation-rmse:7.58908
[11]	validation-rmse:7.43669
[12]	validation-rmse:7.30705
[13]	validation-rmse:7.19679
[14]	validation-rmse:7.10368
[15]	validation-rmse:7.02341
[16]	validation-rmse:6.95495
[17]	validation-rmse:6.89586
[18]	validation-rmse:6.84619
[19]	validation-rmse:6.80220
[20]	validation-rmse:6.76455
[21]	validation-rmse:6.73297
[22]	validation-rmse:6.70498
[23]	validation-rmse:6.68007
[24]	validation-rmse:6.65944
[25]	validation-rmse:6.64053
[26]	validation-rmse:6.62476
[27]	validation-rmse:6.61039
[28]	validation-rmse:6.59774
[29]	validation-rmse:6.58651


2026/11/08 17:03:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/08 17:04:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run clean-snipe-607 at: http://localhost:5000/#/experiments/1/runs/f9f9c7b91530424683e343b46b9acbd5
🧪 View experiment at: http://localhost:5000/#/experiments/1
